# 04. 최종 학습 (8개 증상) + 점수표 (A100, 모델 하나당 약 40분)

구토·오심을 합친 8개 증상으로 KoELECTRA를 학습한다(인프 프로젝트와 같은 모델·설정).

**두 모델**
- **정리 모델**: 03에서 고친 Train 라벨로 학습, 삭제한 칸은 loss에서 뺌 (이 프로젝트의 최종 모델, 시연용으로 저장)
- **원본 모델**: 원래 Train 라벨 그대로 학습 (비교용, `TRAIN_ORIGINAL = False`면 생략)

**채점 기준 세 가지**
- **정리 Val**: 03에서 Train과 같은 규칙으로 고친 Val, 삭제한 칸은 채점에서 뺌 (**대표 점수**)
- **원본 Val**: 원래 라벨 그대로 (남들과 비교되는 점수)
- **고친 Val**: 대회 때 Claude가 고친 Val (파일이 있을 때만, 예전 0.815와 비교)

| 단계 | 하는 일 | 쓰는 함수 |
|---|---|---|
| 0 | 준비 | |
| 1 | 입력 문자열 + 원래 라벨 + 고친 라벨·마스크 | `data.prep`, `data.symptom_matrix`, `cleaning.load_fixed` |
| 2 | 토큰 조각 나누기 | `model.encode_chunks` |
| 3 | **학습 → 모델 저장 → Val 확률 저장** (끝난 모델은 건너뜀) | `trainer.train_model`, `infer.save_model`, `trainer.predict` |
| 4 | **점수표** | `evaluate.score_table` |
| 5 | 증상별 재현율·정밀도 | `metrics.pr_report` |
| 6 | **실험기록.csv**에 한 줄 추가 (나중 실험과 비교) | `evaluate.log_experiment` |

결과: `models/`, `results/04_final/` (모델별 `*_loss.csv` = 에폭별 Loss, `실험기록.csv` = 실험 비교표)

## 0. 준비

In [ ]:
# 드라이브 연결 + src 폴더를 import할 수 있게 경로 추가
from google.colab import drive
drive.mount("/content/drive")

import os, sys
PROJECT_ROOT = "/content/drive/MyDrive/증상인식_최종_프로젝트"   # 폴더를 다른 곳에 두었으면 여기만 바꾸기
os.environ["SYMPTOM_FINAL_ROOT"] = PROJECT_ROOT
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
# src/*.py를 고친 뒤에는 런타임 → 세션 다시 시작 후 이 셀부터 다시 실행

In [ ]:
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from src import config
from src.config import TrainConfig
from src.data import load_labels, prep, symptom_matrix
from src.cleaning import load_fixed
from src.model import encode_chunks
from src.trainer import seed_all, train_model, predict, get_device
from src.infer import save_model
from src.evaluate import claude_fixed_val, score_table, log_experiment
from src.metrics import pr_report

cfg = TrainConfig()
TRAIN_ORIGINAL = True      # 원본 모델(비교용)도 학습할지. 시간이 없으면 False
OUT = f"{config.RESULTS_DIR}/04_final"
os.makedirs(OUT, exist_ok=True)
os.makedirs(config.MODEL_DIR, exist_ok=True)
seed_all(cfg.seed)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

## 1. 입력 + 원래 라벨 + 고친 라벨·마스크

In [ ]:
tok = AutoTokenizer.from_pretrained(cfg.plm_name)
tr_df, va_df = load_labels()
tr = prep(tr_df, tok.sep_token, drop_empty=True)
va = prep(va_df, tok.sep_token, drop_empty=False)
Y_tr0, Y_va0 = symptom_matrix(tr["symptom"]), symptom_matrix(va["symptom"])   # 원래 라벨
Y_tr, M_tr = load_fixed(f"{config.DATA_DIR}/train_fixed.npz", tr)               # 고친 라벨 + 삭제 마스크
Y_va, M_va = load_fixed(f"{config.DATA_DIR}/val_fixed.npz", va)
for n, Y0, Y, M in [("Train", Y_tr0, Y_tr, M_tr), ("Val", Y_va0, Y_va, M_va)]:
    print(f"{n} {Y.shape}: 수정 {int((Y != Y0).sum()):,}칸, 삭제 {int((M == 0).sum()):,}칸")
print(pd.DataFrame({"Train 원래 있음": Y_tr0.sum(0), "Train 최종 있음": (Y_tr * M_tr).sum(0),
                    "Val 원래 있음": Y_va0.sum(0), "Val 최종 있음": (Y_va * M_va).sum(0)},
                   index=config.LABELS).astype(int).to_string())

## 2. 토큰 조각 나누기 (약 5분)

In [ ]:
ch_tr = [encode_chunks(t, tok, cfg.max_len, cfg.stride) for t in tqdm(tr["text"], desc="Train")]
ch_va = [encode_chunks(t, tok, cfg.max_len, cfg.stride) for t in tqdm(va["text"], desc="Val")]

## 3. 학습 → 모델 저장 → Val 확률 저장
모델과 확률은 드라이브에 저장한다. 끊기면 0단계부터 다시 실행하면 끝난 모델은 건너뛴다.

In [ ]:
def run(name, Y, M):
    prob_path = f"{OUT}/{name}_val_prob.npy"
    if os.path.exists(prob_path):
        print(name, "이미 끝남 → 건너뜀")
        return np.load(prob_path)
    seed_all(cfg.seed)
    model = train_model(ch_tr, Y, cfg, tok, M, desc=name, loss_csv=f"{OUT}/{name}_loss.csv")
    save_model(model, f"{config.MODEL_DIR}/{name}.pt", config.LABELS, cfg)
    p = predict(model, ch_va, tok, desc=f"{name} Val")
    np.save(prob_path, p)
    del model; torch.cuda.empty_cache()
    return p

probs = {"정리 모델": run("정리모델", Y_tr, M_tr)}
if TRAIN_ORIGINAL:
    probs["원본 모델"] = run("원본모델", Y_tr0, None)

## 4. 점수표 (Macro F1, 기준값 0.5)

In [ ]:
val_sets = {"정리 Val": (Y_va, M_va), "원본 Val": (Y_va0, None)}
fixed = claude_fixed_val(va_df)
if fixed is not None:
    val_sets["고친 Val"] = fixed
macro, detail = score_table(probs, val_sets)
macro.to_csv(f"{OUT}/점수_macro.csv", encoding="utf-8-sig")
detail.to_csv(f"{OUT}/점수_증상별.csv", encoding="utf-8-sig")
display(macro)
display(detail)

## 5. 증상별 재현율·정밀도 (정리 모델, 정리 Val)
재현율이 특히 낮은 증상(예: 두통)이 있으면, 그 증상 통화 오버샘플링 같은 보완 실험을 하나씩 추가로 검토한다.

In [ ]:
pr = pr_report(Y_va, probs["정리 모델"], config.LABELS, 0.5, M_va)
pr.to_csv(f"{OUT}/재현율_정밀도.csv", encoding="utf-8-sig")
display(pr)

## 6. 실험기록.csv에 남기기
실험마다 한 줄: 실험 이름, 바꾼 설정, Val 기준별 Macro F1, 증상별 F1·재현율·정밀도.
나중에 오버샘플링 등 실험을 돌릴 때는 `run`의 이름과 아래 `SETTINGS`만 바꿔 같은 파일에 줄을 늘린다(같은 이름이면 덮어씀).

In [ ]:
SETTINGS = {"정리 모델": "Qwen 수정 라벨 + 애매한 칸 삭제, 기본 설정", "원본 모델": "원래 라벨, 기본 설정"}
for m in probs:
    det = {}
    for v, (Y, M) in val_sets.items():
        t = pr_report(Y, probs[m], config.LABELS, 0.5, M)
        t["F1"] = detail[f"{m} / {v}"].reindex(t.index)
        det[v] = t
    log = log_experiment(f"{OUT}/실험기록.csv", m, SETTINGS.get(m, ""), macro.loc[m].to_dict(), det)
display(log[[c for c in log.columns if "Macro" in c or c in ("실험", "바꾼 설정")]])